In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # to passs you historical records/conversations
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory # in built memory in langchain
from langchain_core.messages import HumanMessage, AIMessage 
from langchain_core.runnables.history import RunnableWithMessageHistory
load_dotenv()


True

In [10]:
llm = ChatOpenAI(model="gpt-4o-mini")


# exploration of in memory class

history = InMemoryChatMessageHistory()


In [11]:
# this is just for exploraton 
history.add_user_message('hi')
history.add_ai_message('hello')

In [13]:
history.messages

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content='hello', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [14]:
prompt = ChatPromptTemplate.from_messages([
    ("system","You are a professional email support agent"),
    MessagesPlaceholder(variable_name="history"), # slot for past message
    ("human","{input}") # current user message

    ])

prompt

ChatPromptTemplate(input_variables=['history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_c

## Complete solutions


In [22]:
# step 1 : defined your propt
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify complaints as: Billing / Technical / General.
Priority: High (financial impact > $100 or urgent) / Medium / Low.
Remember all customer details throughout the conversation."""),
    MessagesPlaceholder(variable_name="history"),  # ← past turns injected here
    ("human", "{input}"),                           # ← new message here
])


# step 2 :
chain = prompt | llm | StrOutputParser()



# step 3

store = {}


def get_seesion_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]



# step 4 :

chat_bot = RunnableWithMessageHistory(

   chain,
   get_seesion_history,
   input_messages_key="input",
   history_messages_key="history"

)


In [23]:
# lets test our solutions

cfg = {"configurable":{"session_id":"123rahul"}}

r1 = chat_bot.invoke({"input":"hi, my name is rahul"},config=cfg)


In [25]:
r2 = chat_bot.invoke({"input":"hi, what is my name"},config=cfg)
print(r2)


Your name is Rahul. How can I help you today?


In [26]:
store

{'123rahul': InMemoryChatMessageHistory(messages=[HumanMessage(content='hi, my name is rahul', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Rahul! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='hi, what is my name', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Rahul. How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}

In [27]:
cfg = {"configurable":{"session_id":"123shubham"}}
r3 = chat_bot.invoke({"input":"hi, what is my name"},config=cfg)
r3



"I'm sorry, but I don't have access to your personal information or previous interactions. Could you please provide your name so I can assist you better?"

In [28]:
store

{'123rahul': InMemoryChatMessageHistory(messages=[HumanMessage(content='hi, my name is rahul', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Rahul! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='hi, what is my name', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Rahul. How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]),
 '123shubham': InMemoryChatMessageHistory(messages=[HumanMessage(content='hi, what is my name', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm sorry, but I don't have access to your personal information or previous interactions. Could you please provide your name so I can assist you better?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}